In [1]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_models, get_features, ModelTypes
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

flash attention installed


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv', 'dv2', 'dv2_b', 'dv3', 'vit_b', 'vit_b_in', 'sam_b', 'deit', 'clip_b', 'eva02_b')
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'dv2_cb', 'dv2_b', 'dvt', 'alibi_dv2', 'alibi_dv2_h', 'alibi_dv2_cb')

models = get_models(enabled_models, '../../trained_models', device=DEVICE, conf_path='../../dinov3')

In [3]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features = {k: [] for k in models.keys()}
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    for i, (model_name, model) in enumerate(models.items()):

        channel_blank = 'cb' in model_name
        feats = get_features(model, img, device=DEVICE, channel_blank=channel_blank, channel_last=True)
        features[model_name].append(feats)

In [4]:
ramp: RampTypes = 'lr+ud'
ramps_to_results: dict[ModelTypes, list[LinearProbeResult]] = {m: [] for m in enabled_models}
MASK_CUTOFF_FRAC = 1
STEP = 4
RANDOM_MASK = True

for model_name in enabled_models:
    for i in range(n_imgs):
        feats = features[model_name][i]
        result = do_linear_probe(feats, ramp, probe_by_channel=False, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        ramps_to_results[model_name].append(result)

In [5]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)


    if results[0]['per_channel_scores'] is None:
        return None, None, mean_score, std_score, mean_pred

    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [6]:
for model_name in enabled_models:
    results = average_results(ramps_to_results[model_name])
    print(f"{model_name}: {results[2]:.4f} ± {results[3]:.4f}")

dv2: 0.9066 ± 0.0281
dv2_cb: 0.8747 ± 0.0369
dv2_b: 0.8472 ± 0.0377
dvt: 0.8514 ± 0.0544
alibi_dv2: -0.1848 ± 0.2335
alibi_dv2_h: -0.2273 ± 0.2775
alibi_dv2_cb: -0.0102 ± 0.1922
